# Session 1: Basics of Deep Learning, Modern Paradigms and PyTorch

Deep learning is a subfield of machine learning that uses neural networks with multiple layers to learn hierarchical representations of data. Rather than hand-engineering features, deep learning models discover useful patterns directly from raw inputs such as images, text, or audio. This ability to learn representations end-to-end has driven breakthroughs in computer vision, natural language processing, speech recognition, and many other domains.

**PyTorch** is an open-source deep learning framework developed by Meta AI that has become the dominant tool in both research and industry. Its appeal comes from a simple design philosophy: tensors as the universal data structure, automatic differentiation for computing gradients, and a Pythonic imperative programming model that makes debugging and experimentation natural. Unlike frameworks that require you to define a static computation graph before running it, PyTorch builds the graph dynamically as your code executes, which means you can use standard Python control flow, print statements, and debuggers throughout your model.

In this session we will build a working understanding of three foundational pillars:

1. **Introduction to PyTorch** — tensors, automatic differentiation, model definition with `nn.Module`, and the training loop.
2. **Training Logistic Regression with Gradient Descent** — building and training a softmax classifier on the MNIST handwritten digit dataset.
3. **Overfitting and Regularization** — understanding the generalization gap and learning practical techniques to control it.

By the end of this notebook you will have trained a working classifier from scratch and developed the intuition needed to understand more complex architectures in future sessions.

## 0. Setup

We begin by importing the libraries we will use throughout this session. **PyTorch** (`torch`) provides the tensor data structure, automatic differentiation engine, and neural network building blocks. **torchvision** supplies standard datasets like MNIST along with common image transformations. **Matplotlib** is our plotting library for visualizations, and **NumPy** serves as a bridge when we need to convert tensors to arrays for plotting.

We also configure the compute device. Modern GPUs can perform thousands of matrix operations in parallel, which makes them dramatically faster than CPUs for training neural networks. The cell below automatically detects whether a CUDA-capable GPU is available and sets the device accordingly.

In [ ]:
%matplotlib inline

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## 1. PyTorch Tensors — The Building Block

At the heart of PyTorch is the **tensor**, a multi-dimensional array that generalizes familiar mathematical objects. A scalar is a 0-dimensional tensor, a vector is 1-dimensional, a matrix is 2-dimensional, and higher-order tensors extend this pattern to any number of axes. For example, a color image is naturally represented as a 3D tensor with dimensions for height, width, and color channels, while a batch of images becomes a 4D tensor.

PyTorch tensors look and feel like NumPy arrays — they support the same slicing, indexing, and element-wise arithmetic — but they carry two critical advantages for deep learning:

- **Device flexibility.** A tensor can live on the CPU or be moved to a GPU with a single method call. GPU-resident tensors participate in massively parallel computation, which is essential for training at scale.
- **Gradient tracking.** When you create a tensor with `requires_grad=True`, PyTorch records every operation performed on it. This recorded history forms a computational graph that the autograd engine later uses to compute derivatives automatically.

Every tensor carries three key attributes: its **shape** describing the size along each dimension, its **dtype** specifying the numerical type of its elements, and its **device** indicating whether it lives on CPU or GPU memory.

In [ ]:
# Creating tensors from data and from random distributions
a = torch.tensor([1.0, 2.0, 3.0])           # 1D tensor from a list
b = torch.randn(3, 4)                        # 2D tensor, standard normal
c = torch.zeros(2, 3, 4)                     # 3D tensor of zeros

print('a:', a)
print(f'  shape={a.shape}, dtype={a.dtype}, device={a.device}')
print(f'b shape={b.shape}')
print(f'c shape={c.shape}')

### 1.1 Reshaping, Indexing, and Slicing

Reshaping tensors is one of the most frequent operations in deep learning. The `view` method returns a tensor with a different shape but sharing the same underlying data, so it is virtually free. The `reshape` method does the same but can handle non-contiguous memory when necessary. A common pattern is flattening an image tensor from shape `[1, 28, 28]` into a vector of shape `[784]` before feeding it to a linear layer.

In [ ]:
x = torch.arange(12).float()
print('Original:', x, '  shape:', x.shape)

x_2d = x.view(3, 4)
print('\nReshaped to (3,4):')
print(x_2d)

x_3d = x.view(2, 2, 3)
print('\nReshaped to (2,2,3):')
print(x_3d)

print('\nIndexing x_2d[1, 2] =', x_2d[1, 2].item())
print('Slicing  x_2d[:, 1:3] =')
print(x_2d[:, 1:3])

### 1.2 CPU vs GPU

Deep learning workloads consist primarily of matrix multiplications and element-wise operations over large tensors. A modern GPU contains thousands of small cores designed for exactly this kind of data-parallel arithmetic, making it 10 to 100 times faster than a CPU for typical training tasks.

Moving tensors between devices is straightforward. The `.to(device)` method returns a copy of the tensor on the target device. When performing operations, all participating tensors must reside on the same device — PyTorch will raise an error if you try to add a CPU tensor to a GPU tensor, which is a helpful safeguard against accidental performance pitfalls.

In [ ]:
x_cpu = torch.randn(2, 3)
x_dev = x_cpu.to(device)
print('CPU device :', x_cpu.device)
print('Target device:', x_dev.device)

### 1.3 Broadcasting — Vectorized Math Without Loops

Broadcasting is a powerful mechanism that allows PyTorch to perform arithmetic between tensors of different shapes without explicitly copying data. The rules are straightforward: dimensions are compared from the trailing end, and two dimensions are compatible if they are equal or if one of them is 1. A dimension of size 1 is "stretched" to match the other.

For example, adding a vector of shape `(3,)` to a matrix of shape `(5, 3)` broadcasts the vector across all five rows. This eliminates the need for explicit loops and keeps your code concise, readable, and fast.

In [ ]:
M = torch.randn(5, 3)
v = torch.randn(3)
result = M + v

print(f'M shape:      {M.shape}')
print(f'v shape:      {v.shape}')
print(f'result shape: {result.shape}  (v was broadcast across rows)')

The visualization below makes this concrete. The left panel shows a 3x3 matrix, the middle panel shows a vector that will be broadcast, and the right panel shows the result. Each cell is annotated with its value so you can verify the addition element by element.

In [ ]:
M_small = torch.tensor([[1., 2., 3.],
                         [4., 5., 6.],
                         [7., 8., 9.]])
v_small = torch.tensor([10., 20., 30.])
out_small = M_small + v_small

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ['Matrix M', 'Vector v (broadcast)', 'M + v']
data = [M_small, v_small.unsqueeze(0).expand(3, -1), out_small]

for ax, title, d in zip(axes, titles, data):
    im = ax.imshow(d.numpy(), cmap='YlOrRd', aspect='equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{d[i, j]:.0f}', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='black')
    ax.grid(False)

plt.tight_layout()
plt.show()

---

## 2. Automatic Differentiation — PyTorch Does Calculus For You

Training a neural network requires computing the gradient of a loss function with respect to every learnable parameter. Doing this by hand would be tedious and error-prone for models with millions of parameters. PyTorch solves this with **autograd**, its automatic differentiation engine.

The key idea is the **computational graph**. When you perform operations on tensors that have `requires_grad=True`, PyTorch silently builds a directed acyclic graph that records every operation along with the inputs and outputs. Each node in this graph knows how to compute its own local derivative. When you call `.backward()` on a scalar output, the engine traverses this graph in reverse — from the loss back to the parameters — applying the chain rule at each step to accumulate gradients.

This approach is distinct from both **symbolic differentiation**, which manipulates mathematical expressions and can produce unwieldy formulas, and **numerical differentiation** using finite differences, which is slow and susceptible to floating-point errors. Autograd computes exact derivatives at machine precision with a cost proportional to the forward pass, making it the practical engine behind all modern deep learning.

### 2.1 A Simple Example

Consider the function $f(x) = x^2 + 2x + 1$. Its analytical derivative is $f'(x) = 2x + 2$. We will compute this at $x = 3$ using autograd and verify it matches the hand-calculated result.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
f = x**2 + 2*x + 1
f.backward()

print(f'x          = {x.item()}')
print(f'f(x)       = {f.item()}')
print(f'df/dx (autograd)  = {x.grad.item()}')
print(f'df/dx (analytic)  = {(2*x.detach() + 2).item()}')

### 2.2 Multi-Variable Gradients

Autograd naturally handles functions of multiple variables. Given $g(a, b) = a^2 b + b^3$, the partial derivatives are $\partial g / \partial a = 2ab$ and $\partial g / \partial b = a^2 + 3b^2$. A single call to `.backward()` computes both gradients simultaneously.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
g = a**2 * b + b**3
g.backward()

print(f'g(a={a.item()}, b={b.item()}) = {g.item()}')
print(f'dg/da (autograd) = {a.grad.item()},  analytic 2ab = {(2*a.detach()*b.detach()).item()}')
print(f'dg/db (autograd) = {b.grad.item()},  analytic a^2+3b^2 = {(a.detach()**2 + 3*b.detach()**2).item()}')

### 2.3 Visualizing Gradients as Slopes

The gradient at a point tells you the slope of the function at that point. Geometrically, it defines a tangent line. The plot below shows $f(x) = x^2 + 2x + 1$ along with the tangent line at $x = 3$, whose slope is the gradient $f'(3) = 8$.

In [ ]:
x0 = 3.0
slope = 2 * x0 + 2
y0 = x0**2 + 2*x0 + 1
xs = torch.linspace(-5, 6, 300)
ys = xs**2 + 2*xs + 1
tangent = y0 + slope * (xs - x0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs.numpy(), ys.numpy(), linewidth=2, label=r'$f(x) = x^2 + 2x + 1$')
ax.plot(xs.numpy(), tangent.numpy(), '--', linewidth=2, label=f'Tangent at x={x0:.0f}, slope={slope:.0f}')
ax.scatter([x0], [y0], s=80, zorder=5, color='red')
ax.annotate(f'  ({x0:.0f}, {y0:.0f})\n  gradient = {slope:.0f}',
            xy=(x0, y0), fontsize=11, color='red')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_ylim(-5, 40)
ax.legend(fontsize=11)
ax.set_title('Gradient as the Slope of the Tangent Line', fontsize=14)
plt.tight_layout()
plt.show()

---

## 3. Gradient Descent — Learning by Following the Slope Downhill

Gradient descent is the workhorse optimization algorithm behind virtually all deep learning. The idea is elegantly simple: if you want to minimize a function, compute its gradient at your current position and take a step in the opposite direction. Repeating this process drives the parameters toward a minimum.

Formally, given a loss function $L(\theta)$ and a current parameter value $\theta_t$, the update rule is:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\partial L}{\partial \theta}$$

where $\eta$ is the **learning rate**, a hyperparameter that controls the step size. Choosing the right learning rate is critical:

- **Too small:** The optimizer makes tiny steps and convergence is painfully slow. Training may stall long before reaching a good solution.
- **Too large:** The optimizer overshoots the minimum and may oscillate wildly or diverge entirely, with the loss increasing instead of decreasing.
- **Just right:** The optimizer converges smoothly to a minimum in a reasonable number of steps.

We demonstrate this below on a simple 1D loss function $L(\theta) = (\theta - 4)^2$, whose minimum is at $\theta = 4$.

In [ ]:
def run_gd(lr, steps=60):
    theta = torch.tensor(-6.0, requires_grad=True)
    history = []
    for step in range(steps):
        loss = (theta - 4)**2
        loss.backward()
        with torch.no_grad():
            theta -= lr * theta.grad
        theta.grad.zero_()
        history.append((step, theta.item(), loss.item()))
    return history

configs = [
    (0.01, 'lr = 0.01 (too slow)'),
    (0.10, 'lr = 0.10 (good)'),
    (0.55, 'lr = 0.55 (unstable)'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, label in configs:
    hist = run_gd(lr)
    steps_arr = [h[0] for h in hist]
    losses = [h[2] for h in hist]
    thetas = [h[1] for h in hist]
    axes[0].plot(steps_arr, losses, linewidth=2, label=label)
    axes[1].plot(steps_arr, thetas, linewidth=2, label=label)

axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title(r'Loss $L(\theta) = (\theta - 4)^2$ Over Time', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(-5, 110)

axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel(r'$\theta$', fontsize=12)
axes[1].set_title(r'Parameter $\theta$ Over Time', fontsize=13)
axes[1].axhline(4.0, linestyle='--', color='gray', alpha=0.7, label='optimum (4.0)')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

The left panel shows how the loss evolves. With `lr=0.01` the loss decreases too gradually. With `lr=0.10` convergence is smooth and rapid. With `lr=0.55` the optimizer oscillates and struggles to settle. The right panel tells the same story from the perspective of the parameter value itself.

---

## 4. Defining Models with `nn.Module`

PyTorch provides the `nn.Module` base class as the standard way to define neural network models. Every custom model inherits from `nn.Module` and implements two methods:

- **`__init__(self)`** — Declare the layers and learnable parameters of your model. Each layer you assign as an attribute is automatically registered, which means PyTorch can enumerate all parameters when you call `model.parameters()`.
- **`forward(self, x)`** — Define the computation that transforms an input tensor into an output. This is where you wire together the layers declared in `__init__` and apply activation functions.

You never call `forward` directly. Instead, you call the model object as a function — `output = model(input)` — and PyTorch dispatches to `forward` internally while handling hooks and other bookkeeping.

Three additional patterns appear constantly:

- **`model.parameters()`** returns an iterator over all learnable tensors, which you pass to an optimizer.
- **`model.to(device)`** moves every parameter to the specified device in one call.
- **`model.train()` / `model.eval()`** toggle training and evaluation modes, which affects layers like dropout and batch normalization.

Below is a minimal example that defines a model with a single hidden layer.

In [ ]:
class TinyModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        return self.layer2(x)

demo_model = TinyModel(input_dim=4, hidden_dim=8, output_dim=2).to(device)
print(demo_model)
print()
for name, param in demo_model.named_parameters():
    print(f'{name:20s} shape={str(list(param.shape)):12s} device={param.device}')

---

## 5. Training Logistic Regression on MNIST

With tensors, autograd, gradient descent, and `nn.Module` in our toolkit, we are ready to train a real model on real data. We will build a **softmax regression** classifier — the multi-class generalization of logistic regression — and train it on the MNIST dataset of handwritten digits.

### 5.0 From Sigmoid to Softmax

In binary classification, the **sigmoid function** $\sigma(z) = \frac{1}{1 + e^{-z}}$ maps a single real-valued score, called a logit, to a probability between 0 and 1. When there are $K > 2$ classes, we generalize to the **softmax function**, which takes a vector of $K$ logits and produces a probability distribution:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

The outputs are non-negative and sum to 1, so they form a valid probability distribution over classes.

The **cross-entropy loss** measures how far the model's predicted distribution is from the true label. For a true class $y$ and predicted logits $z$:

$$L = -\log\left(\frac{e^{z_y}}{\sum_j e^{z_j}}\right)$$

Intuitively, cross-entropy penalizes the model heavily when it assigns low probability to the correct class. PyTorch's `nn.CrossEntropyLoss` expects **raw logits** rather than probabilities — it applies log-softmax internally for numerical stability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid
zs = torch.linspace(-8, 8, 400)
sig = torch.sigmoid(zs)
axes[0].plot(zs.numpy(), sig.numpy(), linewidth=2, color='#2196F3')
axes[0].axhline(0.5, linestyle='--', color='gray', alpha=0.5)
axes[0].axvline(0.0, linestyle='--', color='gray', alpha=0.5)
axes[0].set_xlabel('Logit z', fontsize=12)
axes[0].set_ylabel(r'$\sigma(z)$', fontsize=12)
axes[0].set_title('Sigmoid Function (binary)', fontsize=13)

# Softmax
logits_example = torch.tensor([2.0, 1.0, 0.5, -1.0])
probs = F.softmax(logits_example, dim=0)
classes = ['Class 0', 'Class 1', 'Class 2', 'Class 3']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
bars = axes[1].bar(classes, probs.numpy(), color=colors, edgecolor='black', linewidth=0.5)
for bar, p in zip(bars, probs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{p:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_title(f'Softmax of logits {logits_example.tolist()}', fontsize=13)
axes[1].set_ylim(0, 0.6)
axes[1].grid(axis='x', visible=False)

plt.tight_layout()
plt.show()

### 5.1 The MNIST Data Pipeline

The MNIST dataset contains 60,000 training images and 10,000 test images of handwritten digits, each a 28x28 grayscale image. It is the canonical introductory dataset for image classification.

Before feeding images to a model, we apply a **transform pipeline**. The `ToTensor()` transform converts a PIL image to a float tensor with values in $[0, 1]$. The `Normalize(mean, std)` transform then shifts and scales the pixel values so that they have approximately zero mean and unit variance. This normalization helps gradient-based optimization converge faster because the loss landscape becomes more symmetric and well-conditioned.

The `DataLoader` wraps a dataset and provides an iterator that yields mini-batches. The `shuffle=True` option randomizes the order of examples each epoch, which is important because training on the same fixed order can introduce subtle biases. The `batch_size` controls how many examples are processed together in one forward-backward pass.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

x_sample, y_sample = next(iter(train_loader))
print(f'Batch images: shape={x_sample.shape}, dtype={x_sample.dtype}')
print(f'Batch labels: shape={y_sample.shape}, dtype={y_sample.dtype}')
print(f'Training set:  {len(train_ds)} images')
print(f'Test set:      {len(test_ds)} images')

### 5.2 Visualizing the Data

It is always good practice to visually inspect your data before training. Below we display a grid of sample images from the training set, un-normalizing them back to the original pixel range for display.

In [ ]:
def unnormalize(img):
    return img * 0.3081 + 0.1307

x_vis, y_vis = next(iter(train_loader))
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = unnormalize(x_vis[i]).squeeze(0)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{y_vis[i].item()}', fontsize=11)
    ax.axis('off')
plt.suptitle('Sample MNIST Training Images', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.3 Building the Softmax Regression Model

Our model is the simplest possible classifier: a single linear layer. Each 28x28 image is flattened into a 784-dimensional vector, and the linear layer maps it to 10 logits, one per digit class. Despite its simplicity, this model typically reaches over 90% accuracy on MNIST, establishing a useful baseline.

We pair the model with the cross-entropy loss and the **SGD optimizer with momentum**. Momentum accelerates convergence by maintaining a running average of past gradients, which smooths out noisy updates and helps the optimizer push through flat regions of the loss surface.

In [ ]:
class LinearMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(28 * 28, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.linear(x)

model = LinearMNIST().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

### 5.4 The Training Loop — The Core PyTorch Skill

The training loop is the central rhythm of deep learning in PyTorch. Every iteration follows five steps:

1. **Forward pass.** Feed a batch of inputs through the model to obtain predictions. This is where the computational graph is built.

2. **Compute the loss.** Compare the model's predictions to the true labels using the loss function. The result is a scalar tensor that represents how wrong the model is.

3. **Zero the gradients** with `optimizer.zero_grad()`. PyTorch accumulates gradients by default — each call to `.backward()` *adds* to the existing `.grad` attributes rather than replacing them. If we skip this step, gradients from previous iterations pile up and the updates become meaningless.

4. **Backward pass** with `loss.backward()`. The autograd engine traverses the computational graph in reverse, computing the gradient of the loss with respect to every parameter that has `requires_grad=True`.

5. **Update the parameters** with `optimizer.step()`. The optimizer reads the accumulated gradients and adjusts each parameter according to its update rule.

We also define an `evaluate` function that runs the model on a dataset without computing gradients, using the `@torch.no_grad()` decorator to disable gradient tracking and reduce memory usage.

In [ ]:
@torch.no_grad()
def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)              # Step 1: forward pass
        loss = loss_fn(logits, y)       # Step 2: compute loss

        optimizer.zero_grad()           # Step 3: zero gradients
        loss.backward()                 # Step 4: backward pass
        optimizer.step()                # Step 5: update parameters

        b = x.size(0)
        total_loss += loss.item() * b
        total_acc  += accuracy_from_logits(logits.detach(), y) * b
        n += b
    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)

        b = x.size(0)
        total_loss += loss.item() * b
        total_acc  += accuracy_from_logits(logits, y) * b
        n += b
    return total_loss / n, total_acc / n

### 5.5 Training the Model

We train for 5 epochs. An **epoch** is one complete pass through the entire training set. After each epoch we evaluate on the held-out test set to track generalization.

In [ ]:
epochs = 5
train_hist, test_hist = [], []

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
    te_loss, te_acc = evaluate(model, test_loader, loss_fn)
    train_hist.append((tr_loss, tr_acc))
    test_hist.append((te_loss, te_acc))
    print(f'Epoch {epoch:02d} | '
          f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | '
          f'Test Loss: {te_loss:.4f}  Acc: {te_acc:.4f}')

### 5.6 Training Curves

Plotting the loss and accuracy over epochs is the most basic diagnostic tool. A healthy training run shows loss decreasing and accuracy increasing on both the training and test sets. If the training loss drops but the test loss stagnates or rises, that is a signal of overfitting — a topic we will explore in the next section.

In [ ]:
train_losses = [t[0] for t in train_hist]
train_accs   = [t[1] for t in train_hist]
test_losses  = [t[0] for t in test_hist]
test_accs    = [t[1] for t in test_hist]
epoch_range  = list(range(1, epochs + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epoch_range, train_losses, 'o-', linewidth=2, label='Train Loss')
axes[0].plot(epoch_range, test_losses,  's--', linewidth=2, label='Test Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Loss Over Training', fontsize=13)
axes[0].set_xticks(epoch_range)
axes[0].legend(fontsize=11)

axes[1].plot(epoch_range, train_accs, 'o-', linewidth=2, label='Train Accuracy')
axes[1].plot(epoch_range, test_accs,  's--', linewidth=2, label='Test Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy Over Training', fontsize=13)
axes[1].set_xticks(epoch_range)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Final Test Accuracy: {test_accs[-1]:.4f}')

### 5.7 Confusion Matrix

The confusion matrix reveals which digit pairs the model tends to confuse. Each row represents the true label and each column represents the predicted label. A perfect classifier would have all counts on the diagonal. Off-diagonal entries highlight systematic errors — for example, the model might confuse 4s and 9s because they share similar stroke patterns.

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, loader, num_classes=10):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    model.eval()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        for t, p in zip(y.view(-1), preds.view(-1)):
            cm[t.long(), p.long()] += 1
    return cm.cpu()

cm = compute_confusion_matrix(model, test_loader)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm.numpy(), cmap='Blues')

for i in range(10):
    for j in range(10):
        val = cm[i, j].item()
        color = 'white' if val > cm.max().item() * 0.5 else 'black'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=9, color=color)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix on Test Set', fontsize=14)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
fig.colorbar(im, ax=ax, shrink=0.8)
ax.grid(False)
plt.tight_layout()
plt.show()

---

## 6. Overfitting and Regularization

A model **overfits** when it memorizes the training data rather than learning patterns that generalize to unseen examples. The telltale symptom is a growing gap between training performance and test performance: the training loss continues to drop while the test loss stagnates or increases.

This is fundamentally a question of **model capacity vs. data complexity**. A model with many parameters relative to the amount of training data has enough flexibility to memorize individual examples, including their noise and idiosyncrasies. On the other hand, a model that is too simple may **underfit** — it lacks the capacity to capture the true underlying patterns and performs poorly on both training and test data.

The goal of machine learning is **generalization**: performing well on data the model has never seen during training. The gap between training and test performance is called the **generalization gap**, and managing it is one of the central challenges in practice. **Regularization** is the family of techniques designed to reduce overfitting without fundamentally changing the model architecture.

### 6.1 Demonstrating Overfitting

The clearest way to see overfitting in action is to train a model on a very small subset of the data. With only 200 training examples but 7,850 parameters, our linear model has far more capacity than it needs to memorize the training set. We train for 30 epochs and track both training and test performance.

In [ ]:
torch.manual_seed(42)

small_subset = Subset(train_ds, range(200))
small_loader = DataLoader(small_subset, batch_size=64, shuffle=True)

def train_and_track(model, train_ldr, test_ldr, optimizer, loss_fn, epochs):
    """Train a model and return per-epoch train/test loss and accuracy."""
    train_hist, test_hist = [], []
    for _ in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_ldr, optimizer, loss_fn)
        te_loss, te_acc = evaluate(model, test_ldr, loss_fn)
        train_hist.append((tr_loss, tr_acc))
        test_hist.append((te_loss, te_acc))
    return train_hist, test_hist

overfit_model = LinearMNIST().to(device)
overfit_opt = torch.optim.SGD(overfit_model.parameters(), lr=0.1, momentum=0.9)
overfit_epochs = 30

of_train, of_test = train_and_track(
    overfit_model, small_loader, test_loader, overfit_opt, loss_fn, overfit_epochs
)

print(f'Final Train Acc: {of_train[-1][1]:.4f}')
print(f'Final Test Acc:  {of_test[-1][1]:.4f}')

In [ ]:
ep = list(range(1, overfit_epochs + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep, [t[0] for t in of_train], 'o-', markersize=4, linewidth=2, label='Train Loss')
axes[0].plot(ep, [t[0] for t in of_test],  's--', markersize=4, linewidth=2, label='Test Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Overfitting: Loss Curves (200 training examples)', fontsize=13)
axes[0].legend(fontsize=11)

axes[1].plot(ep, [t[1] for t in of_train], 'o-', markersize=4, linewidth=2, label='Train Accuracy')
axes[1].plot(ep, [t[1] for t in of_test],  's--', markersize=4, linewidth=2, label='Test Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Overfitting: Accuracy Curves (200 training examples)', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

The training accuracy climbs toward 100% as the model memorizes the small training set, but the test accuracy plateaus much lower. The growing gap between the two curves is the hallmark of overfitting.

### 6.2 L2 Regularization (Weight Decay)

One of the most widely used regularization techniques is **L2 regularization**, also known as **weight decay**. The idea is to add a penalty to the loss function that discourages the model from assigning large values to its weights:

$$L_{\text{regularized}} = L_{\text{original}} + \lambda \|\mathbf{w}\|^2$$

where $\lambda$ is a hyperparameter controlling the strength of the penalty. Large weights allow the model to be overly sensitive to small variations in the input, which is a common mechanism of overfitting. By penalizing large weights, L2 regularization encourages the model to find simpler solutions that rely on broader patterns rather than memorized details.

In PyTorch, L2 regularization is implemented directly in the optimizer via the `weight_decay` parameter. Setting `weight_decay=0.01` is equivalent to adding $0.01 \|\mathbf{w}\|^2$ to the loss.

We now repeat the small-subset experiment with three different weight decay settings to observe the effect.

In [ ]:
torch.manual_seed(42)

wd_configs = [
    (0.0,  'No regularization'),
    (0.01, r'$\lambda$ = 0.01'),
    (0.1,  r'$\lambda$ = 0.10'),
]

wd_results = {}
for wd, label in wd_configs:
    torch.manual_seed(42)
    m = LinearMNIST().to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9, weight_decay=wd)
    tr_h, te_h = train_and_track(m, small_loader, test_loader, opt, loss_fn, overfit_epochs)
    wd_results[label] = (tr_h, te_h)
    print(f'{label:25s} | Final Train Acc: {tr_h[-1][1]:.4f}  Test Acc: {te_h[-1][1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#F44336', '#2196F3', '#4CAF50']
ep = list(range(1, overfit_epochs + 1))

for (label, (tr_h, te_h)), color in zip(wd_results.items(), colors):
    axes[0].plot(ep, [t[0] for t in tr_h], '--', color=color, alpha=0.5, linewidth=1.5)
    axes[0].plot(ep, [t[0] for t in te_h], '-',  color=color, linewidth=2, label=label)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Test Loss with Different Weight Decay\n(dashed = train)', fontsize=13)
axes[0].legend(fontsize=10)

for (label, (tr_h, te_h)), color in zip(wd_results.items(), colors):
    axes[1].plot(ep, [t[1] for t in tr_h], '--', color=color, alpha=0.5, linewidth=1.5)
    axes[1].plot(ep, [t[1] for t in te_h], '-',  color=color, linewidth=2, label=label)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Test Accuracy with Different Weight Decay\n(dashed = train)', fontsize=13)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

### 6.3 Early Stopping

**Early stopping** is a simple but effective regularization strategy. The idea is to monitor the validation loss during training and stop when it begins to increase, even if the training loss is still decreasing. The epoch at which validation loss is minimized typically represents the best trade-off between underfitting and overfitting.

In the plot below, we annotate the epoch at which the test loss was lowest during our overfitting experiment. Stopping training at that point would have given us the best generalization performance.

In [ ]:
test_losses_of = [t[0] for t in of_test]
train_losses_of = [t[0] for t in of_train]
best_epoch = int(np.argmin(test_losses_of)) + 1
best_loss = test_losses_of[best_epoch - 1]

fig, ax = plt.subplots(figsize=(10, 5))
ep = list(range(1, overfit_epochs + 1))

ax.plot(ep, train_losses_of, 'o-', markersize=4, linewidth=2, label='Train Loss', color='#2196F3')
ax.plot(ep, test_losses_of,  's-', markersize=4, linewidth=2, label='Test Loss', color='#F44336')
ax.axvline(best_epoch, linestyle='--', color='#4CAF50', linewidth=2, alpha=0.8,
           label=f'Early stopping (epoch {best_epoch})')
ax.scatter([best_epoch], [best_loss], s=120, color='#4CAF50', zorder=5, edgecolor='black')
ax.annotate(f'  Best test loss = {best_loss:.4f}',
            xy=(best_epoch, best_loss), fontsize=11, color='#4CAF50')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Early Stopping: Stop When Validation Loss Begins to Rise', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Best epoch for early stopping: {best_epoch}')
print(f'Test accuracy at that epoch: {of_test[best_epoch - 1][1]:.4f}')

### 6.4 The Big Picture: Data Size, Regularization, and Generalization

Beyond weight decay and early stopping, several other regularization techniques are commonly used in deep learning:

- **Dropout** randomly sets a fraction of neuron activations to zero during training, forcing the network to develop redundant representations.
- **Data augmentation** artificially expands the training set by applying random transformations such as rotations, crops, and flips.
- **Batch normalization** normalizes intermediate activations, which has an implicit regularizing effect.

However, the single most effective way to combat overfitting is simply to **use more training data**. The experiment below compares models trained on the small subset vs. the full dataset, illustrating how additional data dramatically reduces the generalization gap.

In [ ]:
torch.manual_seed(42)

configs_final = {
    '200 examples, no reg.': (small_loader, 0.0),
    '200 examples, wd=0.01': (small_loader, 0.01),
    'Full data, no reg.':    (train_loader, 0.0),
    'Full data, wd=0.01':   (train_loader, 0.01),
}

summary_results = {}
for name, (ldr, wd) in configs_final.items():
    torch.manual_seed(42)
    m = LinearMNIST().to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9, weight_decay=wd)
    n_ep = 20
    tr_h, te_h = train_and_track(m, ldr, test_loader, opt, loss_fn, n_ep)
    summary_results[name] = (tr_h, te_h)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
color_map = {'200 examples, no reg.': '#F44336',
             '200 examples, wd=0.01': '#FF9800',
             'Full data, no reg.':    '#2196F3',
             'Full data, wd=0.01':   '#4CAF50'}

for name, (tr_h, te_h) in summary_results.items():
    c = color_map[name]
    ep = list(range(1, len(tr_h) + 1))
    axes[0, 0].plot(ep, [t[0] for t in tr_h], '-',  color=c, linewidth=2, label=name)
    axes[0, 1].plot(ep, [t[0] for t in te_h], '-',  color=c, linewidth=2, label=name)
    axes[1, 0].plot(ep, [t[1] for t in tr_h], '-',  color=c, linewidth=2, label=name)
    axes[1, 1].plot(ep, [t[1] for t in te_h], '-',  color=c, linewidth=2, label=name)

axes[0, 0].set_title('Train Loss', fontsize=13)
axes[0, 1].set_title('Test Loss', fontsize=13)
axes[1, 0].set_title('Train Accuracy', fontsize=13)
axes[1, 1].set_title('Test Accuracy', fontsize=13)

for ax in axes.flat:
    ax.set_xlabel('Epoch', fontsize=11)
    ax.legend(fontsize=9)

axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[1, 0].set_ylabel('Accuracy', fontsize=11)

plt.suptitle('Impact of Data Size and Regularization on Generalization',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n--- Final Test Accuracies ---')
for name, (tr_h, te_h) in summary_results.items():
    print(f'  {name:30s}  Test Acc = {te_h[-1][1]:.4f}')

The summary above demonstrates the key insights:

1. **More data is the best regularizer.** Training on the full dataset virtually eliminates the generalization gap, even without explicit regularization.
2. **Weight decay helps when data is limited.** On the small subset, adding weight decay narrows the gap between training and test performance.
3. **Early stopping provides a complementary safeguard.** By monitoring validation loss, you can halt training before the model begins to overfit.

These principles extend directly to deeper and more complex models, where the same tension between capacity and generalization plays out at a larger scale.

---

## Summary

In this session we covered the foundational building blocks of deep learning with PyTorch:

- **Tensors** are the universal data structure, supporting GPU acceleration and automatic differentiation.
- **Autograd** computes gradients automatically by recording operations in a computational graph and traversing it in reverse.
- **Gradient descent** minimizes a loss function by iteratively stepping in the direction opposite the gradient, with the learning rate controlling step size.
- **`nn.Module`** provides the standard way to define models, encapsulating learnable parameters and the forward computation.
- **The training loop** follows five steps at every iteration: forward pass, compute loss, zero gradients, backward pass, update parameters.
- **Logistic regression** applied to MNIST demonstrates how these pieces come together to train a working classifier from scratch.
- **Overfitting** occurs when a model memorizes training data rather than learning generalizable patterns, and **regularization** techniques like weight decay and early stopping help control it.

These fundamentals form the foundation for everything that follows. In subsequent sessions we will build on this foundation to explore deeper architectures and more powerful training strategies.